In [29]:
import pandas as pd

In [31]:
train = pd.read_csv("../Data/Worldwide Travel Cities Dataset (Ratings and Climate).csv")
train.head()

,id,city,country,region,short_description,latitude,longitude,avg_temp_monthly,ideal_durations,budget_level,culture,adventure,nature,beaches,nightlife,cuisine,wellness,urban,seclusion
0,c54acf38-3029-496b-8c7a-8343ad82785c,Milan,Italy,europe,"Chic streets lined with fashion boutiques, his...",45.464194,9.189635,"{""1"":{""avg"":3.7,""max"":7.8,""min"":0.4},""2"":{""avg...","[""Short trip"",""One week""]",Luxury,5,2,2,1,4,5,3,5,2
1,0bd12654-ed64-424e-a044-7bc574bcf078,Yasawa Islands,Fiji,oceania,"Crystal-clear waters, secluded beaches, and vi...",-17.290947,177.125786,"{""1"":{""avg"":28,""max"":30.8,""min"":25.8},""2"":{""av...","[""Long trip"",""One week""]",Luxury,2,4,5,5,2,3,4,1,5
2,73036cda-9134-46fc-a2c6-807782d59dfb,Whistler,Canada,north_america,Snow-capped peaks and lush forests create a se...,50.117190,-122.954302,"{""1"":{""avg"":-2.5,""max"":0.4,""min"":-5.5},""2"":{""a...","[""Short trip"",""Weekend"",""One week""]",Luxury,3,5,5,2,3,3,4,2,4
3,3872c9c0-6b6e-49e1-9743-f46bfe591b86,Guanajuato,Mexico,north_america,Winding cobblestone streets and colorful facad...,20.987700,-101.000000,"{""1"":{""avg"":15.5,""max"":22.8,""min"":8.7},""2"":{""a...","[""Weekend"",""One week"",""Short trip""]",Mid-range,5,3,3,1,3,4,3,4,2
4,e1ebc1b6-8798-422d-847a-22016faff3fd,Surabaya,Indonesia,asia,Bustling streets filled with the aroma of loca...,-7.245972,112.737827,"{""1"":{""avg"":28.1,""max"":32.5,""min"":25.5},""2"":{""...","[""Short trip"",""Weekend""]",Budget,4,3,3,2,3,4,3,4,2


In [32]:
import json

# 각 row의 avg_temp_monthly 컬럼을 JSON으로 파싱해서 월별 평균 기온만 추출
def extract_avg_temp(json_str):
    temp_dict = json.loads(json_str)
    return [temp_dict[str(i)]["avg"] for i in range(1, 13)]

# 12개 컬럼로 확장
temp_df = train['avg_temp_monthly'].dropna().apply(extract_avg_temp).apply(pd.Series)
temp_df.columns = [f'temp_month_{i}' for i in range(1, 13)]

In [4]:
# 문자열 리스트 → 실제 리스트로 변환
train['ideal_durations_list'] = train['ideal_durations'].dropna().apply(lambda x: json.loads(x))

# One-hot 인코딩을 위한 explode
durations_exploded = train[['id', 'ideal_durations_list']].explode('ideal_durations_list')
duration_ohe = pd.get_dummies(durations_exploded['ideal_durations_list'])

# id 기준으로 다시 집계 (각 도시별)
duration_encoded = durations_exploded[['id']].join(duration_ohe).groupby('id').sum().reset_index()

In [33]:
train.head()

,id,city,country,region,short_description,latitude,longitude,avg_temp_monthly,ideal_durations,budget_level,culture,adventure,nature,beaches,nightlife,cuisine,wellness,urban,seclusion
0,c54acf38-3029-496b-8c7a-8343ad82785c,Milan,Italy,europe,"Chic streets lined with fashion boutiques, his...",45.464194,9.189635,"{""1"":{""avg"":3.7,""max"":7.8,""min"":0.4},""2"":{""avg...","[""Short trip"",""One week""]",Luxury,5,2,2,1,4,5,3,5,2
1,0bd12654-ed64-424e-a044-7bc574bcf078,Yasawa Islands,Fiji,oceania,"Crystal-clear waters, secluded beaches, and vi...",-17.290947,177.125786,"{""1"":{""avg"":28,""max"":30.8,""min"":25.8},""2"":{""av...","[""Long trip"",""One week""]",Luxury,2,4,5,5,2,3,4,1,5
2,73036cda-9134-46fc-a2c6-807782d59dfb,Whistler,Canada,north_america,Snow-capped peaks and lush forests create a se...,50.117190,-122.954302,"{""1"":{""avg"":-2.5,""max"":0.4,""min"":-5.5},""2"":{""a...","[""Short trip"",""Weekend"",""One week""]",Luxury,3,5,5,2,3,3,4,2,4
3,3872c9c0-6b6e-49e1-9743-f46bfe591b86,Guanajuato,Mexico,north_america,Winding cobblestone streets and colorful facad...,20.987700,-101.000000,"{""1"":{""avg"":15.5,""max"":22.8,""min"":8.7},""2"":{""a...","[""Weekend"",""One week"",""Short trip""]",Mid-range,5,3,3,1,3,4,3,4,2
4,e1ebc1b6-8798-422d-847a-22016faff3fd,Surabaya,Indonesia,asia,Bustling streets filled with the aroma of loca...,-7.245972,112.737827,"{""1"":{""avg"":28.1,""max"":32.5,""min"":25.5},""2"":{""...","[""Short trip"",""Weekend""]",Budget,4,3,3,2,3,4,3,4,2


In [34]:
# train과 월별 기온 데이터(temp_df) 병합
train = train.join(temp_df)

# train과 duration one-hot 데이터 병합 (id 기준)
train = train.merge(duration_encoded, on='id', how='left')

train.head()

,id,city,country,region,short_description,latitude,longitude,avg_temp_monthly,ideal_durations,budget_level,...,temp_month_8,temp_month_9,temp_month_10,temp_month_11,temp_month_12,Day trip,Long trip,One week,Short trip,Weekend
0,c54acf38-3029-496b-8c7a-8343ad82785c,Milan,Italy,europe,"Chic streets lined with fashion boutiques, his...",45.464194,9.189635,"{""1"":{""avg"":3.7,""max"":7.8,""min"":0.4},""2"":{""avg...","[""Short trip"",""One week""]",Luxury,...,25.2,20.8,15.2,8.8,4.7,0,0,2,2,0
1,0bd12654-ed64-424e-a044-7bc574bcf078,Yasawa Islands,Fiji,oceania,"Crystal-clear waters, secluded beaches, and vi...",-17.290947,177.125786,"{""1"":{""avg"":28,""max"":30.8,""min"":25.8},""2"":{""av...","[""Long trip"",""One week""]",Luxury,...,25.5,26.3,26.0,27.1,28.2,0,2,2,0,0
2,73036cda-9134-46fc-a2c6-807782d59dfb,Whistler,Canada,north_america,Snow-capped peaks and lush forests create a se...,50.117190,-122.954302,"{""1"":{""avg"":-2.5,""max"":0.4,""min"":-5.5},""2"":{""a...","[""Short trip"",""Weekend"",""One week""]",Luxury,...,17.8,14.5,7.6,1.8,-1.8,0,0,3,3,3
3,3872c9c0-6b6e-49e1-9743-f46bfe591b86,Guanajuato,Mexico,north_america,Winding cobblestone streets and colorful facad...,20.987700,-101.000000,"{""1"":{""avg"":15.5,""max"":22.8,""min"":8.7},""2"":{""a...","[""Weekend"",""One week"",""Short trip""]",Mid-range,...,20.6,19.9,19.0,17.7,14.9,0,0,3,3,3
4,e1ebc1b6-8798-422d-847a-22016faff3fd,Surabaya,Indonesia,asia,Bustling streets filled with the aroma of loca...,-7.245972,112.737827,"{""1"":{""avg"":28.1,""max"":32.5,""min"":25.5},""2"":{""...","[""Short trip"",""Weekend""]",Budget,...,28.4,29.2,29.8,29.6,28.7,0,0,0,2,2


In [35]:
train = train.drop(['avg_temp_monthly', 'ideal_durations'], axis=1)

train.head()

,id,city,country,region,short_description,latitude,longitude,budget_level,culture,adventure,...,temp_month_8,temp_month_9,temp_month_10,temp_month_11,temp_month_12,Day trip,Long trip,One week,Short trip,Weekend
0,c54acf38-3029-496b-8c7a-8343ad82785c,Milan,Italy,europe,"Chic streets lined with fashion boutiques, his...",45.464194,9.189635,Luxury,5,2,...,25.2,20.8,15.2,8.8,4.7,0,0,2,2,0
1,0bd12654-ed64-424e-a044-7bc574bcf078,Yasawa Islands,Fiji,oceania,"Crystal-clear waters, secluded beaches, and vi...",-17.290947,177.125786,Luxury,2,4,...,25.5,26.3,26.0,27.1,28.2,0,2,2,0,0
2,73036cda-9134-46fc-a2c6-807782d59dfb,Whistler,Canada,north_america,Snow-capped peaks and lush forests create a se...,50.117190,-122.954302,Luxury,3,5,...,17.8,14.5,7.6,1.8,-1.8,0,0,3,3,3
3,3872c9c0-6b6e-49e1-9743-f46bfe591b86,Guanajuato,Mexico,north_america,Winding cobblestone streets and colorful facad...,20.987700,-101.000000,Mid-range,5,3,...,20.6,19.9,19.0,17.7,14.9,0,0,3,3,3
4,e1ebc1b6-8798-422d-847a-22016faff3fd,Surabaya,Indonesia,asia,Bustling streets filled with the aroma of loca...,-7.245972,112.737827,Budget,4,3,...,28.4,29.2,29.8,29.6,28.7,0,0,0,2,2


In [36]:
train.region.unique()

array(['europe', 'oceania', 'north_america', 'asia', 'africa',
       'middle_east', 'south_america'], dtype=object)

In [37]:
train = train.drop(['id', 'country', 'short_description'], axis=1)

train.head()

,city,region,latitude,longitude,budget_level,culture,adventure,nature,beaches,nightlife,...,temp_month_8,temp_month_9,temp_month_10,temp_month_11,temp_month_12,Day trip,Long trip,One week,Short trip,Weekend
0,Milan,europe,45.464194,9.189635,Luxury,5,2,2,1,4,...,25.2,20.8,15.2,8.8,4.7,0,0,2,2,0
1,Yasawa Islands,oceania,-17.290947,177.125786,Luxury,2,4,5,5,2,...,25.5,26.3,26.0,27.1,28.2,0,2,2,0,0
2,Whistler,north_america,50.117190,-122.954302,Luxury,3,5,5,2,3,...,17.8,14.5,7.6,1.8,-1.8,0,0,3,3,3
3,Guanajuato,north_america,20.987700,-101.000000,Mid-range,5,3,3,1,3,...,20.6,19.9,19.0,17.7,14.9,0,0,3,3,3
4,Surabaya,asia,-7.245972,112.737827,Budget,4,3,3,2,3,...,28.4,29.2,29.8,29.6,28.7,0,0,0,2,2


In [38]:
train.info

<bound method DataFrame.info of                city         region   latitude   longitude budget_level  \
0             Milan         europe  45.464194    9.189635       Luxury   
1    Yasawa Islands        oceania -17.290947  177.125786       Luxury   
2          Whistler  north_america  50.117190 -122.954302       Luxury   
3        Guanajuato  north_america  20.987700 -101.000000    Mid-range   
4          Surabaya           asia  -7.245972  112.737827       Budget   
..              ...            ...        ...         ...          ...   
555            Maun         africa -19.986095   23.422435    Mid-range   
556      Gothenburg         europe  57.707233   11.967017    Mid-range   
557      Manchester         europe  53.479489   -2.245115    Mid-range   
558      Copenhagen         europe  55.686724   12.570072    Mid-range   
559           Sucre  south_america -19.047725  -65.259431       Budget   

     culture  adventure  nature  beaches  nightlife  ...  temp_month_8  \
0    

In [39]:
train.shape

(560, 31)

In [40]:
#  범주형 데이터 숫자로 변환

train['region'] = train['region'].map({
    'europe': 0,
    'oceania': 1,
    'north_america': 2,
    'asia': 3,
    'africa': 4,
    'south_america': 5,
    'middle_east': 6
})

train['budget_level'] = train['budget_level'].map({
    'Budget': 0,
    'Mid-range': 1,
    'Luxury': 2
})

train.head()

,city,region,latitude,longitude,budget_level,culture,adventure,nature,beaches,nightlife,...,temp_month_8,temp_month_9,temp_month_10,temp_month_11,temp_month_12,Day trip,Long trip,One week,Short trip,Weekend
0,Milan,0,45.464194,9.189635,2,5,2,2,1,4,...,25.2,20.8,15.2,8.8,4.7,0,0,2,2,0
1,Yasawa Islands,1,-17.290947,177.125786,2,2,4,5,5,2,...,25.5,26.3,26.0,27.1,28.2,0,2,2,0,0
2,Whistler,2,50.117190,-122.954302,2,3,5,5,2,3,...,17.8,14.5,7.6,1.8,-1.8,0,0,3,3,3
3,Guanajuato,2,20.987700,-101.000000,1,5,3,3,1,3,...,20.6,19.9,19.0,17.7,14.9,0,0,3,3,3
4,Surabaya,3,-7.245972,112.737827,0,4,3,3,2,3,...,28.4,29.2,29.8,29.6,28.7,0,0,0,2,2


In [42]:
from geopy.distance import geodesic

# 인천공항 위치
incheon_coords = (37.4602, 126.4407)

# 도시별 위도/경도 정보가 train에 있다면
train['airfare_from_incheon'] = train.apply(
    lambda row: geodesic(incheon_coords, (row['latitude'], row['longitude'])).km * 0.12,  # 0.12 = 1km당 요금 (임의값)
    axis=1
)

train.head()

,city,region,latitude,longitude,budget_level,culture,adventure,nature,beaches,nightlife,...,temp_month_9,temp_month_10,temp_month_11,temp_month_12,Day trip,Long trip,One week,Short trip,Weekend,airfare_from_incheon
0,Milan,0,45.464194,9.189635,2,5,2,2,1,4,...,20.8,15.2,8.8,4.7,0,0,2,2,0,1066.244230
1,Yasawa Islands,1,-17.290947,177.125786,2,2,4,5,5,2,...,26.3,26.0,27.1,28.2,0,2,2,0,0,966.494853
2,Whistler,2,50.117190,-122.954302,2,3,5,5,2,3,...,14.5,7.6,1.8,-1.8,0,0,3,3,3,980.391004
3,Guanajuato,2,20.987700,-101.000000,1,5,3,3,1,3,...,19.9,19.0,17.7,14.9,0,0,3,3,3,1422.885648
4,Surabaya,3,-7.245972,112.737827,0,4,3,3,2,3,...,29.2,29.8,29.6,28.7,0,0,0,2,2,618.015904


In [41]:
import numpy as np
from sklearn.model_selection import train_test_split

In [14]:
# Feature와 Target 분리하기

train_label = train.loc[:,'city']
train_input = train.iloc[:,1:]

train_label.head()

0             Milan
1    Yasawa Islands
2          Whistler
3        Guanajuato
4          Surabaya
Name: city, dtype: object

In [15]:
print(train_input.shape)
print(train_label.shape)

(560, 29)
(560,)


In [16]:
# # Train과 valid 나누기

# train_data, valid_data, train_target, valid_target = \
#   train_test_split(
#     train_input,
#     train_label,
#     random_state=42,
#     test_size=0.2
#   )

In [17]:
# print(train_data.shape)
# print(train_target.shape)
# print(valid_data.shape)
# print(valid_target.shape)

In [18]:
# from sklearn.ensemble import RandomForestClassifier

# # 리스트 포함 컬럼 제거
# if 'ideal_durations_list' in train_data.columns:
#     train_data = train_data.drop(columns=['ideal_durations_list'])
#     valid_data = valid_data.drop(columns=['ideal_durations_list'])

# # RandomForest 재실행
# rf = RandomForestClassifier(
#     n_estimators=200,
#     max_depth=10,
#     min_samples_split=5,
#     random_state=42,
#     n_jobs=-1
# )
# rf.fit(train_data, train_target)

# print("Train Score:", rf.score(train_data, train_target))
# print("Valid Score:", rf.score(valid_data, valid_target))